In [1]:
import json
import pandas as pd
from pathlib import Path


def load_json(path: str | Path) -> dict | list:
    """Memuat file JSON."""
    filepath = Path(path)
    if not filepath.is_file():
        raise FileNotFoundError(f"File JSON tidak ditemukan: {filepath}")

    with open(filepath, "r", encoding="utf-8") as f:
        return json.load(f)


def reformat_unieval(unieval: list) -> dict:
    """Menghitung nilai rata-rata untuk setiap metrik UniEval."""
    if not unieval:
        return {}

    coherence   = [item.get("coherence", 0)    for item in unieval]
    relevance   = [item.get("relevance", 0)     for item in unieval]
    fluency     = [item.get("fluency", 0)       for item in unieval]
    consistency = [item.get("consistency", 0)   for item in unieval]

    return {
        "coherence":   coherence,
        "relevance":   relevance,
        "fluency":     fluency,
        "consistency": consistency,
    }


def get_unieval_score(method_num: str, path: str) -> dict:
    """Memuat skor UniEval dari path yang ditentukan lalu merapihkannya."""
    new_path = Path(f"{method_num}_outputs") / path / "unieval.json"
    unieval_data = load_json(new_path)
    return reformat_unieval(unieval_data)


def get_align_score(method_num: str, path: str) -> dict:
    """Memuat AlignScore dari path yang ditentukan."""
    new_path = Path(f"{method_num}_outputs") / path / "alignscore.json"
    align_score_data = load_json(new_path)
    return {"align_score": align_score_data}


def get_generated(method_num: str, path: str) -> dict:
    """Memuat data generated (prediksi & prompt) dari path yang ditentukan."""
    new_path = Path(f"{method_num}_outputs") / path / "generated.json"
    generated_data = load_json(new_path)
    summary  = [item.get("prediction", "") for item in generated_data]
    dialogue = [item.get("prompt", "")     for item in generated_data]
    return {"summary": summary, "dialogue": dialogue}  # ← tambahan return


def merge_metric_generation(method_num: str, folder: str) -> pd.DataFrame:
    """Menggabungkan semua metrik dan data generated ke dalam satu DataFrame."""
    unieval_mean  = get_unieval_score(method_num, folder)
    align_score   = get_align_score(method_num, folder)
    generated     = get_generated(method_num, folder)

    merged = {
        **unieval_mean,
        **align_score,
        **generated,
    }
    return pd.DataFrame(merged)

In [2]:
df = merge_metric_generation("method3", "bart_margin_part")
df.head()

,coherence,relevance,fluency,consistency,align_score,summary,dialogue
0,0.898447,0.957309,0.935121,0.937874,0.859605,Profile: Traveler looking for high-end accommo...,user: I am looking for a high priced hotel in...
1,0.930305,0.932854,0.804797,0.912566,0.899010,Profile: Traveler looking for a 0-star hotel i...,user: I am looking for a 0 star hotel to stay...
2,0.969789,0.947093,0.937903,0.948345,0.637093,Profile: Budget-conscious traveler looking for...,user: Can you help me find a place to stay? I...
3,0.913296,0.954550,0.756134,0.687631,0.667167,Profile: Traveler planning a trip to Queen’s C...,"user: Hi, I am looking for an attraction call..."
4,0.968118,0.976219,0.693360,0.896408,0.856355,Profile: Traveler planning a trip to the north...,"user: Hi, I am planning a trip and looking fo..."


## Setup: Load kedua model (M3 · margin_part & M3 · finetune)

In [3]:
# Load kedua model utama yang akan dibandingkan
df_mp = merge_metric_generation("method3", "bart_margin_part")
df_ft = merge_metric_generation("method3", "bart_finetune")

# Tambahkan kolom penanda model
df_mp["model"] = "margin_part"
df_ft["model"] = "finetune"

# Pastikan indeks sejajar (dokumen yang sama)
assert len(df_mp) == len(df_ft), "Jumlah instance tidak sama!"
print(f"Total instance: {len(df_mp)}")

SEMANTIC_METRICS = ["coherence", "relevance", "fluency", "consistency", "align_score"]
N = 2  # jumlah contoh per sub-kategori


Total instance: 300


## Helper: Fungsi tampil instance

In [4]:
def show_samples(df_source, indices, label=""):
    """Tampilkan instance terpilih secara ringkas."""
    print(f"{'='*70}")
    print(f"  {label}  (n={len(indices)})")
    print(f"{'='*70}")
    for rank, idx in enumerate(indices, 1):
        row = df_source.loc[idx]
        print(f"\n[#{rank}] Index: {idx}")
        for m in SEMANTIC_METRICS:
            print(f"  {m:<15}: {row[m]:.4f}")
        print(f"\n  DIALOGUE  : {str(row['dialogue'])}...")
        print(f"  SUMMARY   : {str(row['summary'])}...")
        print(f"{'-'*70}")

def show_pair(idx, label=""):
    """Tampilkan pasangan margin_part vs finetune untuk index yang sama."""
    print(f"{'='*70}")
    print(f"  {label}  | Index: {idx}")
    print(f"{'='*70}")
    print(f"\n  DIALOGUE : {str(df_mp.loc[idx, 'dialogue'])}...")
    print()
    for m in SEMANTIC_METRICS:
        mp_val = df_mp.loc[idx, m]
        ft_val = df_ft.loc[idx, m]
        delta  = mp_val - ft_val
        sign   = "+" if delta >= 0 else ""
        print(f"  {m:<15}: margin_part={mp_val:.4f}  finetune={ft_val:.4f}  Δ={sign}{delta:.4f}")
    print(f"\n  SUMMARY margin_part : {str(df_mp.loc[idx, 'summary'])}...")
    print(f"  SUMMARY finetune    : {str(df_ft.loc[idx, 'summary'])}...")
    print(f"{'-'*70}")


---
## KATEGORI 1 — Kasus Sukses

### 1.1 Sukses Menyeluruh — margin_part
> **Kriteria:** coherence > 0.90 AND relevance > 0.90 AND consistency > 0.92  
> **Tujuan:** Tunjukkan kondisi ideal di mana model margin menghasilkan ringkasan berkualitas tinggi di semua dimensi.

In [5]:
mask_11 = (
    (df_mp["coherence"]   > 0.90) &
    (df_mp["relevance"]   > 0.90) &
    (df_mp["consistency"] > 0.92)
)
sample_11 = (
    df_mp[mask_11]
    .assign(_score=lambda d: d[SEMANTIC_METRICS].mean(axis=1))
    .nlargest(N, "_score")
    .drop(columns="_score")
)
print(f"Kandidat memenuhi kriteria 1.1: {mask_11.sum()} instance")
show_samples(df_mp, sample_11.index, "1.1 | Sukses Menyeluruh | margin_part")


Kandidat memenuhi kriteria 1.1: 122 instance
  1.1 | Sukses Menyeluruh | margin_part  (n=2)

[#1] Index: 165
  coherence      : 0.9848
  relevance      : 0.9800
  fluency        : 0.9375
  consistency    : 0.9503
  align_score    : 0.9656

  DIALOGUE  :  user: I'd like a train to Ely that leaves Wednesday.
agent: Easy enough. Where is your departure point?
user: Cambridge. I would like to leave after 15:30 if possible.
agent: TR6745 departs cambridge at 15:50 Wednesday, arriving at Ely at 16:07. Will that one work for you? It's only 4.40 GBP to ride.
user: Great can I book that for 8 people and get a reference number please?
agent: You are booked for 8 seats on TR6745 leaving Cambridge Wednesday at 17:50. Reference # is OCWICAXZ and you can pay at the station. 4.40 pounds each.
user: Thank you. That's is all I need. Goodbye.
agent: Welcome. Have a lovely day....
  SUMMARY   : Profile: Traveler planning a train trip from Cambridge to Ely for a group of 8 people.

Problem: Needed a Wedne

### 1.2 Sukses Leksikal — finetune
> **Kriteria:** align_score > top 10% AND coherence > 0.88  
> **Tujuan:** Tunjukkan kondisi terbaik finetune — ringkasan yang selaras dengan sumber dan koheren.

In [6]:
thr_align_ft = df_ft["align_score"].quantile(0.90)
mask_12 = (
    (df_ft["align_score"] >= thr_align_ft) &
    (df_ft["coherence"]   > 0.88)
)
sample_12 = (
    df_ft[mask_12]
    .assign(_score=lambda d: d[SEMANTIC_METRICS].mean(axis=1))
    .nlargest(N, "_score")
    .drop(columns="_score")
)
print(f"Kandidat memenuhi kriteria 1.2: {mask_12.sum()} instance")
show_samples(df_ft, sample_12.index, "1.2 | Sukses Leksikal | finetune")


Kandidat memenuhi kriteria 1.2: 25 instance
  1.2 | Sukses Leksikal | finetune  (n=2)

[#1] Index: 100
  coherence      : 0.9791
  relevance      : 0.9807
  fluency        : 0.9439
  consistency    : 0.9605
  align_score    : 0.9465

  DIALOGUE  :  user: I am at Club Salsa and I need to book a taxi to leave after 06:15.
agent: Sure! What is your destination?
user: I need to go to the Cambridge arts theatre.
agent: Your booking is all set. A black honda will be picking you up.
user: Wonderful, thanks for your help.
agent: You're welcome. Have a good time!...
  SUMMARY   : User Profile: A user arranging a taxi ride from Club Salsa to Cambridge Arts Theatre.

Problem: The user needs to book a taxi to leave after 06:15 for a specific time and destination (Cambridge Arts Theatre).

Resolution:
- Booked a taxi for the user to go to the Cambridge Arts theatre.
- Confirmed that a black Honda will be assigned to the taxi....
----------------------------------------------------------------------

---
## KATEGORI 2 — Kasus Gagal

### 2.1 Halusinasi Fakta — margin_part
> **Kriteria:** consistency < bottom 10%  
> **Tujuan:** Cari fakta dalam ringkasan yang tidak ada atau bertentangan dengan dokumen sumber.

In [7]:
thr_con_lo_mp = df_mp["consistency"].quantile(0.10)
sample_21 = (
    df_mp[df_mp["consistency"] <= thr_con_lo_mp]
    .nsmallest(N, "consistency")
)
print(f"Threshold consistency bottom 10% (margin_part): {thr_con_lo_mp:.4f}")
show_samples(df_mp, sample_21.index, "2.1 | Halusinasi Fakta | margin_part")


Threshold consistency bottom 10% (margin_part): 0.7673
  2.1 | Halusinasi Fakta | margin_part  (n=2)

[#1] Index: 112
  coherence      : 0.5212
  relevance      : 0.3939
  fluency        : 0.4950
  consistency    : 0.5142
  align_score    : 0.5475

  DIALOGUE  :  user: Hello, I need information about Parkside Police Station, please.
agent: The phone number to the police station is 01223358966. Is there anything else I can help you with?
user: Yes, could I get the postcode as well please?
agent: Of course! Their post code is CB11JG.
user: Thank you, that will be all.
agent: Have a great day....
  SUMMARY   : Profile: A user seeking basic contact and location details for Parkside Police Station.

Problem: The user wants the phone number, postcode, and postcode of Parkside police station after receiving the postcode information.  

Resolution:
- Provided the phone numbers: 01223358966 and 01223 358966   - Provided the police station’s postcode: CB1 1JG...
---------------------------------

### 2.2 Halusinasi Fakta — finetune
> **Kriteria:** consistency < bottom 10%  
> **Tujuan:** Bandingkan pola halusinasi finetune vs margin_part — apakah jenis fakta yang salah berbeda?

In [8]:
thr_con_lo_ft = df_ft["consistency"].quantile(0.10)
sample_22 = (
    df_ft[df_ft["consistency"] <= thr_con_lo_ft]
    .nsmallest(N, "consistency")
)
print(f"Threshold consistency bottom 10% (finetune): {thr_con_lo_ft:.4f}")
show_samples(df_ft, sample_22.index, "2.2 | Halusinasi Fakta | finetune")


Threshold consistency bottom 10% (finetune): 0.7726
  2.2 | Halusinasi Fakta | finetune  (n=2)

[#1] Index: 14
  coherence      : 0.1599
  relevance      : 0.0180
  fluency        : 0.4098
  consistency    : 0.5087
  align_score    : 0.5490

  DIALOGUE  :  user: Hi, I'm looking for an expensive hotel in the south of town.
agent: The Lensfield hotel should fit your needs perfectly, would you like for me to set up a booking?
user: Does that hotel have a 3 star rating?
agent: Indeed it does. Does it sound like somewhere you'd like to stay?
user: I do not need any free parking. Is there any expensive hotel in the south with 3 stars and no parking?
agent: No, there is not. The only expensive 3-star hotel in the south is The Lensfield Hotel and it has parking.
user: Okay, I need to book it for seven people 5 nights starting tuesday.
agent: I booked it for you. Your reference number is ALYDOZKZ .
user: Thank you! I also need to book a train to Cambridge.
agent: There are many options. Could y

### 2.3 Ringkasan Tidak Relevan — margin_part
> **Kriteria:** relevance < bottom 10%  
> **Tujuan:** Cari kasus di mana ringkasan menyimpang dari topik utama dokumen sumber.

In [9]:
thr_rel_lo_mp = df_mp["relevance"].quantile(0.10)
sample_23 = (
    df_mp[df_mp["relevance"] <= thr_rel_lo_mp]
    .nsmallest(N, "relevance")
)
print(f"Threshold relevance bottom 10% (margin_part): {thr_rel_lo_mp:.4f}")
show_samples(df_mp, sample_23.index, "2.3 | Ringkasan Tidak Relevan | margin_part")


Threshold relevance bottom 10% (margin_part): 0.5463
  2.3 | Ringkasan Tidak Relevan | margin_part  (n=2)

[#1] Index: 183
  coherence      : 0.0062
  relevance      : 0.0224
  fluency        : 0.6687
  consistency    : 0.7398
  align_score    : 0.8740

  DIALOGUE  :  user: I want a place that serves Swiss food.
agent: I'm sorry that we don't have any swiss restaurants nearby. I could recommend several other restaurants if you would like?
user: Could you please try restaurants that serve Indian food instead?
agent: We have several Indian restaurants. What area of town are you interested in?
user: any. book a table for 8 people at 16:00on tuesday
agent: Okay, I can do that. How about Curry Prince in the East part of town? It is moderately-priced.
user: Yes, that sounds good. Please make the reservation for me.
agent: Booking was successful. The table will be reserved for 15 minutes. Reference number is : E4Q6Z78S . Can I help with anything else?
user: Sure, I'm also looking for a partic

### 2.4 Ringkasan Tidak Koheren — finetune
> **Kriteria:** coherence < bottom 10%  
> **Tujuan:** Cari kasus di mana alur kalimat ringkasan tidak logis atau putus-putus.

In [10]:
thr_coh_lo_ft = df_ft["coherence"].quantile(0.10)
sample_24 = (
    df_ft[df_ft["coherence"] <= thr_coh_lo_ft]
    .nsmallest(N, "coherence")
)
print(f"Threshold coherence bottom 10% (finetune): {thr_coh_lo_ft:.4f}")
show_samples(df_ft, sample_24.index, "2.4 | Ringkasan Tidak Koheren | finetune")


Threshold coherence bottom 10% (finetune): 0.5359
  2.4 | Ringkasan Tidak Koheren | finetune  (n=2)

[#1] Index: 192
  coherence      : 0.0227
  relevance      : 0.0185
  fluency        : 0.7065
  consistency    : 0.7162
  align_score    : 0.7434

  DIALOGUE  :  user: I am planning a trip in Cambridge.
agent: What aspects of it would you like me to help you with?
user: I am looking for a Sunday train to Bishops Stortford please.
agent: There are several leaving on Sunday. What time would you like to depart?
user: The train should leave after 17:15 and should go to bishops stortford.
agent: TR6742 leaves at 17:29 and arrives in bishops stortford at 18:07. Would you like me to book it for you?
user: No need to book. But, can you please give me the travel time and the price of the ticket?
agent: The travel time is 38 minutes and the ticket cost is 8.08 pounds. Is there anything else I can help you with?
user: Yes I am looking for some things to do while I'm in town.
agent: We have almost 

### 2.5 Relevan tapi Tidak Selaras dengan Sumber — finetune
> **Kriteria:** relevance > 0.80 AND align_score < bottom 15%  
> **Tujuan:** Model menghasilkan konten yang terasa relevan secara semantik tetapi menyimpang dari fakta sumber (parafrase terlalu jauh).

In [11]:
thr_align_lo_ft = df_ft["align_score"].quantile(0.15)
mask_25 = (
    (df_ft["relevance"]   > 0.80) &
    (df_ft["align_score"] <= thr_align_lo_ft)
)
sample_25 = (
    df_ft[mask_25]
    .nsmallest(N, "align_score")
)
print(f"Threshold align_score bottom 15% (finetune): {thr_align_lo_ft:.4f}")
print(f"Kandidat: {mask_25.sum()} instance")
show_samples(df_ft, sample_25.index, "2.5 | Relevan tapi Tidak Selaras | finetune")


Threshold align_score bottom 15% (finetune): 0.6020
Kandidat: 32 instance
  2.5 | Relevan tapi Tidak Selaras | finetune  (n=2)

[#1] Index: 115
  coherence      : 0.8616
  relevance      : 0.8146
  fluency        : 0.5259
  consistency    : 0.7590
  align_score    : 0.3528

  DIALOGUE  :  user: Hi, yes, I'm looking for a 4 star hotel to stay in. The only amenity it has to have is free wifi.
agent: Sure, the Cambridge Belfry fits that bill. It has great reviews and is cheap. Can I get more information for you?
user: Does that also have free parking? And I forgot to mention I'd prefer a guesthouse.
agent: In that case, the Cambridge Belfry would not work. It's a hotel, not a guesthouse. The Acorn Guesthouse has wifi, parking, is moderately priced and has 4 stars.
user: Book for me 1 person and 2 nights starting from wednesday.
agent: Your booking was successful.Your reference number is : 6UPHIRT7 . Is there anything else you need help with today?
user: Yes I am looking for someplace to e

### 2.6 Selaras tapi Tidak Relevan — finetune
> **Kriteria:** align_score > top 15% AND relevance < bottom 15%  
> **Tujuan:** Model menyalin frasa sumber dengan baik (align tinggi) namun melewatkan poin utama (relevance rendah).

In [12]:
thr_align_hi_ft = df_ft["align_score"].quantile(0.85)
thr_rel_lo_ft   = df_ft["relevance"].quantile(0.15)
mask_26 = (
    (df_ft["align_score"] >= thr_align_hi_ft) &
    (df_ft["relevance"]   <= thr_rel_lo_ft)
)
sample_26 = (
    df_ft[mask_26]
    .nsmallest(N, "relevance")
)
print(f"Threshold align top 15%: {thr_align_hi_ft:.4f}  |  relevance bottom 15%: {thr_rel_lo_ft:.4f}")
print(f"Kandidat: {mask_26.sum()} instance")
show_samples(df_ft, sample_26.index, "2.6 | Selaras tapi Tidak Relevan | finetune")


Threshold align top 15%: 0.9150  |  relevance bottom 15%: 0.6135
Kandidat: 4 instance
  2.6 | Selaras tapi Tidak Relevan | finetune  (n=2)

[#1] Index: 183
  coherence      : 0.5547
  relevance      : 0.1584
  fluency        : 0.8998
  consistency    : 0.9060
  align_score    : 0.9291

  DIALOGUE  :  user: I want a place that serves Swiss food.
agent: I'm sorry that we don't have any swiss restaurants nearby. I could recommend several other restaurants if you would like?
user: Could you please try restaurants that serve Indian food instead?
agent: We have several Indian restaurants. What area of town are you interested in?
user: any. book a table for 8 people at 16:00on tuesday
agent: Okay, I can do that. How about Curry Prince in the East part of town? It is moderately-priced.
user: Yes, that sounds good. Please make the reservation for me.
agent: Booking was successful. The table will be reserved for 15 minutes. Reference number is : E4Q6Z78S . Can I help with anything else?
user: Su

---
## KATEGORI 3 — Kasus Divergen
> Semua seleksi menggunakan **indeks yang sama** pada kedua model sehingga dokumen sumber identik.

### 3.1 finetune Lebih Selaras, margin_part Lebih Relevan
> **Kriteria:** Δalign (ft−mp) > top 10% AND Δrelevance (mp−ft) > top 10%  
> **Tujuan:** Tunjukkan trade-off utama — finetune lebih dekat ke sumber, margin_part lebih menangkap makna.

In [13]:
delta_align = df_ft["align_score"] - df_mp["align_score"]   # ft lebih tinggi = positif
delta_rel   = df_mp["relevance"]   - df_ft["relevance"]     # mp lebih tinggi = positif

thr_da = delta_align.quantile(0.90)
thr_dr = delta_rel.quantile(0.90)

mask_31 = (delta_align >= thr_da) & (delta_rel >= thr_dr)
indices_31 = delta_rel[mask_31].nlargest(3).index

print(f"Threshold Δalign top 10%: {thr_da:.4f}  |  Δrelevance top 10%: {thr_dr:.4f}")
print(f"Kandidat: {mask_31.sum()} instance")
for idx in indices_31:
    show_pair(idx, "3.1 | Divergen: ft align↑ vs mp relevance↑")


Threshold Δalign top 10%: 0.2090  |  Δrelevance top 10%: 0.4099
Kandidat: 3 instance
  3.1 | Divergen: ft align↑ vs mp relevance↑  | Index: 134

  DIALOGUE :  user: I need a train to stansted airport from cambridge please.
agent: Ok, what day will you be traveling?
user: Thursday. I need to arrive by 12:45.
agent: The closest to thing I can find is a train leaving at 5:40 on Thursday. It arrives at Stansted Airport at 6:08. Is this okay?
user: Yes, that works. Can I get the train id number and the price please?
agent: the train Number is TR7966, it will cost you 10.10 pounds.
user: I need to find me a room as well.
agent: What part of town would you prefer? And, do you need internet?
user: Looking for something in the North, cheap price range please. Don't care much about internet.
agent: There are two guesthouses that meet your criteria. The City Centre North B and B, and the Worth House. Would you like me to book one of those for you?
user: Sure, Saturday for 5 nights and 7 people.
a

### 3.2 margin_part Jauh Lebih Koheren
> **Kriteria:** Δcoherence (mp−ft) > top 10%  
> **Tujuan:** Tunjukkan tipe dokumen di mana margin_part menghasilkan alur ringkasan yang lebih baik.

In [14]:
delta_coh = df_mp["coherence"] - df_ft["coherence"]
thr_dc    = delta_coh.quantile(0.90)

indices_32 = delta_coh[delta_coh >= thr_dc].nlargest(N).index

print(f"Threshold Δcoherence top 10%: {thr_dc:.4f}")
print(f"Kandidat: {(delta_coh >= thr_dc).sum()} instance")
for idx in indices_32:
    show_pair(idx, "3.2 | Divergen: margin_part jauh lebih koheren")


Threshold Δcoherence top 10%: 0.3202
Kandidat: 30 instance
  3.2 | Divergen: margin_part jauh lebih koheren  | Index: 192

  DIALOGUE :  user: I am planning a trip in Cambridge.
agent: What aspects of it would you like me to help you with?
user: I am looking for a Sunday train to Bishops Stortford please.
agent: There are several leaving on Sunday. What time would you like to depart?
user: The train should leave after 17:15 and should go to bishops stortford.
agent: TR6742 leaves at 17:29 and arrives in bishops stortford at 18:07. Would you like me to book it for you?
user: No need to book. But, can you please give me the travel time and the price of the ticket?
agent: The travel time is 38 minutes and the ticket cost is 8.08 pounds. Is there anything else I can help you with?
user: Yes I am looking for some things to do while I'm in town.
agent: We have almost 80 attractions to choose from. Did you have something in mind to do?
user: Yes, I am looking for a museum in the centre.
agent

### 3.3 finetune Kolaps, margin_part Stabil
> **Kriteria:** align_score finetune < bottom 10% AND align_score margin_part > median  
> **Tujuan:** Cari tipe dokumen yang secara khusus membuat finetune gagal total sementara margin_part masih stabil.

In [15]:
thr_align_lo_ft2 = df_ft["align_score"].quantile(0.10)
thr_align_med_mp = df_mp["align_score"].median()

mask_33 = (
    (df_ft["align_score"] <= thr_align_lo_ft2) &
    (df_mp["align_score"] >= thr_align_med_mp)
)
indices_33 = df_ft[mask_33]["align_score"].nsmallest(N).index

print(f"Threshold ft align bottom 10%: {thr_align_lo_ft2:.4f}  |  mp align median: {thr_align_med_mp:.4f}")
print(f"Kandidat: {mask_33.sum()} instance")
for idx in indices_33:
    show_pair(idx, "3.3 | Divergen: finetune kolaps, margin_part stabil")


Threshold ft align bottom 10%: 0.5579  |  mp align median: 0.7743
Kandidat: 8 instance
  3.3 | Divergen: finetune kolaps, margin_part stabil  | Index: 87

  DIALOGUE :  user: Hi, I am looking for a place to stay and it has to have at least a two star rating and free WiFi.
agent: Okay, I found some options to fit your needs on the North side of town and on the east side. Do you have a preference for which area?
user: No, I'm going to have my car with me so I can get to any part of town. But now that I think of it, I'd like to have free parking.
agent: I have three hotels meeting those requirements, two moderately priced and one expensive.
user: I would like one with a 2 star rating, if the moderately priced one is 2 stars I'd like to book it for 2 nights starting Thursday.
agent: How many people would you like for that reservation?
user: I would like to book for 5 people, two nights starting Thursday, if that is not available then Tuesday and the reference number please.
agent: Your boo

### 3.4 Keduanya Gagal Bersamaan
> **Kriteria:** consistency finetune < bottom 10% AND consistency margin_part < bottom 10%  
> **Tujuan:** Identifikasi tipe dokumen yang memang sulit untuk semua model — kemungkinan noise atau struktur dialog yang tidak biasa.

In [16]:
thr_con_lo_both_ft = df_ft["consistency"].quantile(0.10)
thr_con_lo_both_mp = df_mp["consistency"].quantile(0.10)

mask_34 = (
    (df_ft["consistency"] <= thr_con_lo_both_ft) &
    (df_mp["consistency"] <= thr_con_lo_both_mp)
)
# Urutkan berdasarkan rata-rata consistency terendah
combined_con = (df_ft["consistency"] + df_mp["consistency"]) / 2
indices_34   = combined_con[mask_34].nsmallest(N).index

print(f"Threshold consistency bottom 10% — ft: {thr_con_lo_both_ft:.4f}  mp: {thr_con_lo_both_mp:.4f}")
print(f"Kandidat (keduanya gagal): {mask_34.sum()} instance")
for idx in indices_34:
    show_pair(idx, "3.4 | Divergen: keduanya gagal bersamaan")


Threshold consistency bottom 10% — ft: 0.7726  mp: 0.7673
Kandidat (keduanya gagal): 7 instance
  3.4 | Divergen: keduanya gagal bersamaan  | Index: 155

  DIALOGUE :  user: I would like to find a certain restaurant called the Maharajah Tandoori Restaurant. Can you find it for me?
agent: It is located at 41518 Castle Street City Centre.
user: Great. Can you make me a reservation for 5 people at 14:30 on Wednesday.
agent: The booking was successful. The table will be reserved for 15 minutes.Your reference number is : XL60G8AI .
user: Thanks so much. I'm also looking for a place to stay.
agent: We have 33 places to stay in the area. Would you like to narrow down your needs, like price range or area?
user: Well I would like it to be moderately priced, 4 stars and include free wifi and include free parking.
agent: I don't have anything that fits all your specific requests in that area. Would you like to try a different area or would you be willing to change specifics?
user: Surely there's 

---
## Rekap: Daftar Index Terpilih per Sub-kategori

In [17]:
# Jalankan setelah semua cell di atas sudah dieksekusi
recap = {
    "1.1 Sukses Menyeluruh (mp)"         : list(sample_11.index),
    "1.2 Sukses Leksikal (ft)"            : list(sample_12.index),
    "2.1 Halusinasi Fakta (mp)"           : list(sample_21.index),
    "2.2 Halusinasi Fakta (ft)"           : list(sample_22.index),
    "2.3 Tidak Relevan (mp)"              : list(sample_23.index),
    "2.4 Tidak Koheren (ft)"              : list(sample_24.index),
    "2.5 Relevan tapi Tidak Selaras (ft)" : list(sample_25.index),
    "2.6 Selaras tapi Tidak Relevan (ft)" : list(sample_26.index),
    "3.1 ft align↑ vs mp relevance↑"      : list(indices_31),
    "3.2 mp jauh lebih koheren"            : list(indices_32),
    "3.3 ft kolaps, mp stabil"             : list(indices_33),
    "3.4 Keduanya gagal"                  : list(indices_34),
}

print(f"{'Sub-Kategori':<42} {'Indices'}")
print("-" * 70)
for k, v in recap.items():
    print(f"{k:<42} {v}")


Sub-Kategori                               Indices
----------------------------------------------------------------------
1.1 Sukses Menyeluruh (mp)                 [165, 294]
1.2 Sukses Leksikal (ft)                   [100, 95]
2.1 Halusinasi Fakta (mp)                  [112, 155]
2.2 Halusinasi Fakta (ft)                  [14, 46]
2.3 Tidak Relevan (mp)                     [183, 253]
2.4 Tidak Koheren (ft)                     [192, 24]
2.5 Relevan tapi Tidak Selaras (ft)        [115, 87]
2.6 Selaras tapi Tidak Relevan (ft)        [183, 128]
3.1 ft align↑ vs mp relevance↑             [134, 232, 128]
3.2 mp jauh lebih koheren                  [192, 48]
3.3 ft kolaps, mp stabil                   [87, 111]
3.4 Keduanya gagal                         [155, 72]
